In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.input.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.mongodb.output.uri", "mongodb://database:27017/proyecto_bigdata.datos_scraping") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.2") \
    .getOrCreate()

df = spark.read.format("com.mongodb.spark.sql.DefaultSource").load()

df.printSchema()
df.show(5)

root
 |-- _id: struct (nullable = true)
 |    |-- oid: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- identificador: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+-------------------+------------+--------------------+-----+
|                 _id|      fecha_captura|       grupo|       identificador|valor|
+--------------------+-------------------+------------+--------------------+-----+
|{69dd51248255b817...|2026-04-13 20:24:57|G1_Ecommerce|A Light in the Attic|51.77|
|{69dd51248255b817...|2026-04-13 20:24:57|G1_Ecommerce|  Tipping the Velvet|53.74|
|{69dd51248255b817...|2026-04-13 20:24:57|G1_Ecommerce|          Soumission| 50.1|
|{69dd51248255b817...|2026-04-13 20:24:57|G1_Ecommerce|       Sharp Objects|47.82|
|{69dd51248255b817...|2026-04-13 20:24:57|G1_Ecommerce|Sapiens: A Brief ...|54.23|
+--------------------+-------------------+------------+--------------------+-----+
only sho

In [4]:
from pyspark.sql.functions import col

df_filtrado = df.filter(
    (col("identificador").isNotNull()) & (col("valor") > 0)
)

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros válidos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 60
Registros válidos: 60


In [6]:
df.select("identificador", "valor").show()

+--------------------+-----+
|       identificador|valor|
+--------------------+-----+
|A Light in the Attic|51.77|
|  Tipping the Velvet|53.74|
|          Soumission| 50.1|
|       Sharp Objects|47.82|
|Sapiens: A Brief ...|54.23|
|     The Requiem Red|22.65|
|The Dirty Little ...|33.34|
|The Coming Woman:...|17.93|
|The Boys in the B...| 22.6|
|     The Black Maria|52.15|
|Starving Hearts (...|13.99|
|Shakespeare's Son...|20.66|
|         Set Me Free|17.46|
|Scott Pilgrim's P...|52.29|
|Rip it Up and Sta...|35.02|
|Our Band Could Be...|57.25|
|                Olio|23.88|
|Mesaerion: The Be...|37.59|
|Libertarianism fo...|51.33|
|It's Only the Him...|45.17|
+--------------------+-----+
only showing top 20 rows



In [7]:
df.filter(df["valor"] > 100).show()

+---+-------------+-----+-------------+-----+
|_id|fecha_captura|grupo|identificador|valor|
+---+-------------+-----+-------------+-----+
+---+-------------+-----+-------------+-----+



In [8]:
df.groupBy("grupo").count().show()

+------------+-----+
|       grupo|count|
+------------+-----+
|G1_Ecommerce|   60|
+------------+-----+



In [10]:
from pyspark.sql import functions as F

reporte_categorias = df.groupBy("grupo").agg(
    F.count("identificador").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO:")
reporte_categorias.show()

ANALISIS DE MERCADO:
+------------+---------------+-----------------+-------------+-------------+
|       grupo|cantidad_libros|  precio_promedio|precio_minimo|precio_maximo|
+------------+---------------+-----------------+-------------+-------------+
|G1_Ecommerce|             60|35.00266666666666|        12.84|        57.31|
+------------+---------------+-----------------+-------------+-------------+



In [12]:
from pyspark.sql import functions as F

NOMBRE_GRUPO = "G1_Ecommerce"

ticket_salida = df.filter(F.col("grupo") == NOMBRE_GRUPO) \
.groupBy("grupo") \
.agg(
    F.count("identificador").alias("total_libros"),
    F.round(F.avg("valor"), 2).alias("precio_medio"),
    F.max("fecha_captura").alias("ultima_sincronizacion")
)

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()

--- TICKET DE SALIDA: G1_Ecommerce ---
+------------+------------+------------+---------------------+
|       grupo|total_libros|precio_medio|ultima_sincronizacion|
+------------+------------+------------+---------------------+
|G1_Ecommerce|          60|        35.0|  2026-04-13 20:25:06|
+------------+------------+------------+---------------------+

